# QUY TRÌNH 2 - DATA INGESTION

## E-commerce Data Lakehouse trên Object Storage

Mục tiêu:

- Đọc dữ liệu Raw từ Quy trình 1
- Kiểm tra metadata của các file
- Kết nối đến MinIO
- Đưa dữ liệu vào Object Storage
- Ghi log quá trình ingestion

In [5]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from minio import Minio

In [6]:
# Thiết lập đường dẫn thư mục dự án và thư mục dữ liệu thô
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Raw directory: {RAW_DIR}"
    )

print("Project:", BASE_DIR)
print("Raw data:", RAW_DIR)

Project: /Users/camlee05/development/E-commerce-Data-Lakehouse
Raw data: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw


In [3]:
# Cấu hình kết nối MinIO
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print("Đã khởi tạo MinIO client.")

Đã khởi tạo MinIO client.


In [21]:
# Kiểm tra kết nối MinIO và liệt kê các bucket
# Lưu ý: Khởi động MinIO trước khi chạy cell này.
# Chạy tại thư mục gốc của project:
#     docker-compose up -d
# Sau đó kiểm tra MinIO Console tại:
#     http://localhost:9001
try:
    buckets = list(client.list_buckets())

    print("Kết nối MinIO thành công!")
    print("Các bucket hiện có:")

    for bucket in buckets:
        print("-", bucket.name)

except Exception as e:
    print("Không thể kết nối MinIO:")
    print(e)

Kết nối MinIO thành công!
Các bucket hiện có:
- ecommerce-lakehouse


In [22]:
# Tạo bucket 
BUCKET_NAME = "ecommerce-lakehouse"

if not client.bucket_exists(BUCKET_NAME):
    client.make_bucket(BUCKET_NAME)
    print(f"Đã tạo bucket: {BUCKET_NAME}")
else:
    print(f"Bucket đã tồn tại: {BUCKET_NAME}")

Bucket đã tồn tại: ecommerce-lakehouse


In [23]:
# Định nghĩa các dataset và tải dữ liệu từ các file CSV

DATASETS = {
    "customers": RAW_DIR / "customers.csv",
    "products": RAW_DIR / "products.csv",
    "orders": RAW_DIR / "orders.csv",
    "order_items": RAW_DIR / "order_items.csv",
    "reviews": RAW_DIR / "reviews.csv",
    "payments": RAW_DIR / "payments.csv"
}

datasets = {}

for name, path in DATASETS.items():
    datasets[name] = pd.read_csv(path)
    print(f"{name}: {len(datasets[name])} records")

customers: 99441 records
products: 32951 records
orders: 99441 records
order_items: 112650 records
reviews: 99224 records
payments: 103886 records


In [24]:
# Kiểm tra metadata của các file dữ liệu

def get_file_metadata(file_path):
    stat = file_path.stat()

    return {
        "file_name": file_path.name,
        "file_type": file_path.suffix,
        "file_size_bytes": stat.st_size,
        "file_size_mb": round(stat.st_size / (1024 * 1024), 2),
        "created_time": datetime.fromtimestamp(
            stat.st_ctime
        ).isoformat(),
        "modified_time": datetime.fromtimestamp(
            stat.st_mtime
        ).isoformat()
    }

metadata = []

for name, path in DATASETS.items():
    info = get_file_metadata(path)
    info["dataset"] = name
    info["rows"] = len(datasets[name])
    info["columns"] = len(datasets[name].columns)
    
    metadata.append(info)

metadata_df = pd.DataFrame(metadata)

metadata_df

,file_name,file_type,file_size_bytes,file_size_mb,created_time,modified_time,dataset,rows,columns
0,customers.csv,.csv,8562270,8.17,2026-09-24T17:17:29.352765,2026-09-24T17:17:29.352765,customers,99441,5
1,products.csv,.csv,2795930,2.67,2026-09-24T17:17:29.422749,2026-09-24T17:17:29.422749,products,32951,9
2,orders.csv,.csv,17406884,16.60,2026-09-24T17:17:29.613953,2026-09-24T17:17:29.613953,orders,99441,8
3,order_items.csv,.csv,14911576,14.22,2026-09-24T17:17:29.814797,2026-09-24T17:17:29.814797,order_items,112650,7
4,reviews.csv,.csv,13955535,13.31,2026-09-24T17:17:29.982229,2026-09-24T17:17:29.982229,reviews,99224,7
5,payments.csv,.csv,5633362,5.37,2026-09-24T17:17:30.073488,2026-09-24T17:17:30.073488,payments,103886,5


In [25]:
# Kiểm tra schema của các dataset
schema_info = []

for name, df in datasets.items():
    for column in df.columns:
        schema_info.append({
            "dataset": name,
            "column": column,
            "dtype": str(df[column].dtype)
        })

schema_df = pd.DataFrame(schema_info)

schema_df.head(20)

,dataset,column,dtype
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,products,product_id,str
6,products,product_category_name,str
7,products,product_name_lenght,float64
8,products,product_description_lenght,float64
9,products,product_photos_qty,float64


In [26]:
# Upload các file dữ liệu thô lên MinIO
ingestion_logs = []

for name, path in DATASETS.items():

    object_name = f"raw/{path.name}"

    try:
        result = client.fput_object(
            BUCKET_NAME,
            object_name,
            str(path),
            content_type="text/csv"
        )

        file_size = path.stat().st_size

        log = {
            "file": path.name,
            "object": object_name,
            "status": "SUCCESS",
            "size_bytes": file_size,
            "size_mb": round(file_size / (1024 * 1024), 2),
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

        ingestion_logs.append(log)

        print(f"✓ SUCCESS: {path.name}")

    except Exception as e:

        log = {
            "file": path.name,
            "object": object_name,
            "status": "FAILED",
            "size_bytes": path.stat().st_size,
            "size_mb": round(
                path.stat().st_size / (1024 * 1024), 2
            ),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "error": str(e)
        }

        ingestion_logs.append(log)

        print(f"✗ FAILED: {path.name}")
        print(e)

✓ SUCCESS: customers.csv
✓ SUCCESS: products.csv
✓ SUCCESS: orders.csv
✓ SUCCESS: order_items.csv
✓ SUCCESS: reviews.csv
✓ SUCCESS: payments.csv


In [27]:
# Kiểm tra dữ liệu trong bucket MinIO

objects = client.list_objects(
    BUCKET_NAME,
    prefix="raw/",
    recursive=True
)

for obj in objects:
    print(
        obj.object_name,
        "|",
        obj.size,
        "bytes"
    )

raw/customers.csv | 8562270 bytes
raw/order_items.csv | 14911576 bytes
raw/orders.csv | 17406884 bytes
raw/payments.csv | 5633362 bytes
raw/products.csv | 2795930 bytes
raw/reviews.csv | 13955535 bytes


In [28]:
# Ghi log ingestion vào DataFrame
ingestion_log_df = pd.DataFrame(ingestion_logs)

ingestion_log_df

,file,object,status,size_bytes,size_mb,timestamp
0,customers.csv,raw/customers.csv,SUCCESS,8562270,8.17,2026-09-24T10:36:59.765062+00:00
1,products.csv,raw/products.csv,SUCCESS,2795930,2.67,2026-09-24T10:36:59.815953+00:00
2,orders.csv,raw/orders.csv,SUCCESS,17406884,16.60,2026-09-24T10:36:59.861853+00:00
3,order_items.csv,raw/order_items.csv,SUCCESS,14911576,14.22,2026-09-24T10:36:59.902952+00:00
4,reviews.csv,raw/reviews.csv,SUCCESS,13955535,13.31,2026-09-24T10:36:59.939649+00:00
5,payments.csv,raw/payments.csv,SUCCESS,5633362,5.37,2026-09-24T10:36:59.966276+00:00


In [29]:
# Lưu log ingestion vào file JSON
log_data = {
    "pipeline": "ecommerce-data-lakehouse",
    "process": "data_ingestion",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "bucket": BUCKET_NAME,
    "records": ingestion_logs
}

log_file = BASE_DIR / "data" / "ingestion_log.json"

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(
        log_data,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"Đã tạo log: {log_file}")

Đã tạo log: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/ingestion_log.json


In [30]:
# Upload log ingestion lên MinIO
client.fput_object(
    BUCKET_NAME,
    "logs/ingestion_log.json",
    str(log_file),
    content_type="application/json"
)

print("✓ Đã upload ingestion log lên MinIO.")



✓ Đã upload ingestion log lên MinIO.


In [31]:
# Kiểm tra log ingestion trong bucket MinIO
objects = client.list_objects(
    BUCKET_NAME,
    prefix="logs/",
    recursive=True
)

for obj in objects:
    print(obj.object_name)

logs/ingestion_log.json


In [32]:
# Kiểm tra tổng quan ingestion
raw_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="raw/",
        recursive=True
    )
)

success_count = sum(
    1 for log in ingestion_logs
    if log["status"] == "SUCCESS"
)

failed_count = sum(
    1 for log in ingestion_logs
    if log["status"] == "FAILED"
)

print("========== INGESTION SUMMARY ==========")
print("Bucket:", BUCKET_NAME)
print("Tổng số file:", len(DATASETS))
print("Upload thành công:", success_count)
print("Upload thất bại:", failed_count)
print("Số object trong raw:", len(raw_objects))

========== INGESTION SUMMARY ==========
Bucket: ecommerce-lakehouse
Tổng số file: 6
Upload thành công: 6
Upload thất bại: 0
Số object trong raw: 6
